# Machine Sound Fault Detection - Pretrained Embeddings (MIMII)

This is a **second method**. It does not train a copy model.

**How it works**
1. A pretrained audio model (**AST**, trained on AudioSet) turns each clip into a list of numbers (an **embedding**).
2. We learn what **healthy** embeddings look like (training clips only).
3. A clip whose embedding is **far from healthy** = possible fault.

We train nothing. We only measure distance. Faulty clips are never used to build the model or the threshold.

**What you need to do**
1. Add the MIMII dataset (**Add Input**).
2. Turn on the GPU (**Settings > Accelerator > GPU**).
3. Turn on **Internet** (**Settings > Internet**). It is needed once, to download the AST model (about 350 MB).
4. Click **Run All**.

**Time:** my guess is 20 to 60 minutes on a GPU. I have not measured it on Kaggle.

## Words used in this notebook

| Word | Meaning |
|---|---|
| **clip** | One 10-second sound file |
| **embedding** | A list of 768 numbers that describes a clip |
| **Mahalanobis** | A distance that knows how healthy embeddings are spread. Bigger = stranger |
| **kNN** | Distance to the closest healthy training clips. Bigger = stranger |
| **score** | The distance. Higher = more likely faulty |
| **AUC / pAUC** | How well the score separates healthy from faulty. 0.5 = random guess |
| **recall** | Share of faulty clips that we catch |
| **false alarm** | Share of healthy clips we wrongly call faulty |
| **threshold** | Score above which we say "faulty" (95th percentile of healthy validation scores) |

## Step 1 - Import tools

In [ ]:
import glob, gc, re, sys, time, os, subprocess, importlib.util, shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.covariance import LedoitWolf
from sklearn.metrics import (roc_auc_score, roc_curve, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from IPython.display import display

for name in ("librosa", "soundfile", "transformers"):
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", name])
import librosa
import soundfile as sf
from transformers import ASTFeatureExtractor, ASTModel

## Step 2 - Settings

The defaults are fine.

In [ ]:
# ---- Where things are ----
DATA_ROOT = "/kaggle/input"      # the audio files are somewhere inside this folder
WORK_DIR  = "/kaggle/working"    # results are saved here

# ---- What to run ----
MACHINES  = ["fan", "pump", "slider", "valve"]
IDS       = ["id_00", "id_01", "id_02", "id_03", "id_04", "id_05", "id_06"]
SNR       = 6                    # noise level: 6, 0 or -6
MAX_HOURS = 10
LIMIT     = None                 # for a quick test, set a number (e.g. 40) = max clips per run

# ---- Sound / model settings ----
SAMPLE_RATE = 16000
MIC         = 0                  # the dataset has 8 microphones. We use the first one.
MODEL_NAME  = "MIT/ast-finetuned-audioset-10-10-0.4593"
EMB_BATCH   = 16                 # clips per batch on the GPU
NUM_WORKERS = 4                  # CPU threads that read the audio
KNN_K       = 3                  # kNN looks at the 3 closest healthy clips
THRESHOLD_PERCENT = 95
SEED        = 42

# Decided BEFORE looking at any result. Used for the main table.
HEADLINE = ("pool", "mahalanobis")

## Step 3 - Check the GPU

You should see `cuda`. On CPU it works, but it is very slow.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "no GPU")

## Step 4 - Find the audio files

In [ ]:
def read_path(path):
    parts = path.split("/")[::-1]          # start from the file name
    label   = next((p for p in parts if p in ("normal", "abnormal")), None)
    machine = next((p for p in parts if p in ("fan", "pump", "slider", "valve")), None)
    mid     = next((p for p in parts if re.fullmatch(r"id_\d+", p)), None)
    snr = None
    for p in parts:
        m = re.match(r"(-?\d+)\s*_?dB", p)
        if m:
            snr = int(m.group(1))
            break
    return label, machine, mid, snr

rows = []
for path in glob.glob(DATA_ROOT + "/**/*.wav", recursive=True):
    label, machine, mid, snr = read_path(path)
    if label and machine and mid:
        rows.append([path, machine, mid, label, snr])

df = pd.DataFrame(rows, columns=["path", "machine", "id", "label", "snr"])
assert len(df) > 0, "No audio found. Did you add the dataset to this notebook (Add Input)?"
print(len(df), "clips found")

Number of clips per machine, ID, noise level and label:

In [ ]:
counts = df.groupby(["machine", "id", "snr", "label"], dropna=False).size().unstack("label", fill_value=0)
display(counts)

## Step 5 - Choose the runs

One run = one machine + one ID. A run needs enough healthy and faulty clips.

In [ ]:
USE_SNR = df["snr"].notna().any()     # some datasets have no noise level in the folder name

def get_files(machine, mid):
    rows = df[(df["machine"] == machine) & (df["id"] == mid)]
    if USE_SNR:
        rows = rows[rows["snr"] == SNR]
    healthy = sorted(rows[rows["label"] == "normal"]["path"])
    faulty  = sorted(rows[rows["label"] == "abnormal"]["path"])
    if LIMIT:
        healthy, faulty = healthy[:LIMIT], faulty[:max(LIMIT // 2, 6)]
    return healthy, faulty

plan = []
for machine in MACHINES:
    for mid in IDS:
        healthy, faulty = get_files(machine, mid)
        if len(healthy) >= 30 and len(faulty) >= 5:
            plan.append((machine, mid))
        else:
            print("skip", machine, mid, "- not enough clips")

print(len(plan), "runs planned:")
print(plan)

## Step 6 - Read the sound

We read one microphone of each clip.

In [ ]:
def load_audio(path):
    audio, rate = sf.read(path, dtype="float32", always_2d=True)   # shape: (samples, microphones)
    audio = audio[:, min(MIC, audio.shape[1] - 1)]                 # keep one microphone
    if rate != SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=rate, target_sr=SAMPLE_RATE)
    return audio

## Step 7 - Load the AST model

It is downloaded once. We use it only to read clips (no training).

In [ ]:
extractor = ASTFeatureExtractor.from_pretrained(MODEL_NAME)
ast = ASTModel.from_pretrained(MODEL_NAME).to(device).eval()
if device == "cuda":
    ast = ast.half()                       # half precision = faster
print("AST ready. Parameters:", round(sum(p.numel() for p in ast.parameters()) / 1e6), "million")

## Step 8 - Turn clips into embeddings

For each clip we keep 4 kinds of embedding (all 768 numbers):

| Name | Meaning |
|---|---|
| `pool` | The final summary of the clip (our main choice) |
| `l6`, `l9`, `l12` | Average of the model's inner view at layer 6, 9, 12 |

Embeddings are saved to disk, so a restart does not repeat this work.

In [ ]:
FEATURES = ["pool", "l6", "l9", "l12"]
EMB_DIR = WORK_DIR + "/embeddings"
os.makedirs(EMB_DIR, exist_ok=True)

from concurrent.futures import ThreadPoolExecutor
from collections import deque

def prepare(path):
    audio = load_audio(path)
    return extractor(audio, sampling_rate=SAMPLE_RATE, return_tensors="np")["input_values"][0]

@torch.no_grad()
def embed_files(files):
    out = {k: [] for k in FEATURES}
    batches = [files[i:i + EMB_BATCH] for i in range(0, len(files), EMB_BATCH)]
    with ThreadPoolExecutor(NUM_WORKERS) as pool:
        # CPU threads read the next batches while the GPU works on the current one
        waiting = deque(pool.map(prepare, b) for b in batches[:3])
        for k in range(len(batches)):
            x = torch.from_numpy(np.stack(list(waiting.popleft()))).to(device)
            if k + 3 < len(batches):
                waiting.append(pool.map(prepare, batches[k + 3]))
            if device == "cuda":
                x = x.half()
            o = ast(input_values=x, output_hidden_states=True)
            out["pool"].append(o.pooler_output.float().cpu().numpy())
            for layer in (6, 9, 12):
                out[f"l{layer}"].append(o.hidden_states[layer][:, 2:].float().mean(dim=1).cpu().numpy())
    return {k: np.concatenate(v) for k, v in out.items()}

def get_embeddings(machine, mid):
    healthy, faulty = get_files(machine, mid)
    files = healthy + faulty
    cache = f"{EMB_DIR}/{machine}_{mid}_snr{SNR}_n{len(files)}.npz"
    if os.path.exists(cache):
        z = np.load(cache, allow_pickle=True)
        emb = {k: z[k].astype(np.float32) for k in FEATURES}
    else:
        emb = embed_files(files)
        np.savez(cache, **{k: v.astype(np.float16) for k, v in emb.items()})
    return files, emb

## Step 9 - Split the healthy clips

70% train, 10% validation, 20% test. Faulty clips go only to the test set.
The split is by clip, so no clip is in two sets. It is the **same split** as the dense notebook (same seed).

In [ ]:
def split_healthy(healthy_files):
    order = np.random.default_rng(SEED).permutation(len(healthy_files))
    n_train = int(0.7 * len(order))
    n_val   = int(0.1 * len(order))
    train = [healthy_files[i] for i in order[:n_train]]
    val   = [healthy_files[i] for i in order[n_train:n_train + n_val]]
    test  = [healthy_files[i] for i in order[n_train + n_val:]]
    return train, val, test

## Step 10 - Two ways to score a clip

- **Mahalanobis:** distance from the healthy centre, using how healthy embeddings spread.
  We use a *shrinkage* estimate, because we have few clips and 768 numbers.
- **kNN:** average distance to the 3 closest healthy training clips (cosine distance).

In [ ]:
def score_mahalanobis(train, x):
    return LedoitWolf().fit(train).mahalanobis(x)

def score_knn(train, x, k=KNN_K):
    a = train / np.linalg.norm(train, axis=1, keepdims=True)
    b = x / np.linalg.norm(x, axis=1, keepdims=True)
    dist = 1.0 - b @ a.T
    dist.sort(axis=1)
    return dist[:, :k].mean(axis=1)

SCORERS = {"mahalanobis": score_mahalanobis, "knn": score_knn}

## Step 11 - Measure the result

Threshold = 95th percentile of healthy **validation** scores. We never use faulty clips to set it.

Extra numbers for you:
- **recall at 5% false alarm** = share of faults caught if we allow only 5% false alarms (read from the ROC curve).
- **accuracy** and **balanced accuracy** at the threshold.

In [ ]:
def get_metrics(val_scores, test_scores, y):
    threshold = np.percentile(val_scores, THRESHOLD_PERCENT)
    pred = (test_scores > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    fpr, tpr, _ = roc_curve(y, test_scores)
    return {
        "auc":  roc_auc_score(y, test_scores),
        "pauc": roc_auc_score(y, test_scores, max_fpr=0.1),
        "f1":   f1_score(y, pred),
        "recall": tp / (tp + fn),
        "false_alarm": fp / (fp + tn),
        "recall_at_5pct_fa": float(np.interp(0.05, fpr, tpr)),
        "accuracy": (tp + tn) / len(y),
        "bal_acc": 0.5 * (tp / (tp + fn) + tn / (tn + fp)),
        "threshold": threshold,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

In [ ]:
# Published AUC from the MIMII paper (Table 3), for comparison.
# Order inside each tuple: 6 dB, 0 dB, -6 dB
PUBLISHED = {
 "fan":    {"id_00": (.75,.63,.57), "id_01": (.97,.90,.70), "id_02": (.99,.83,.68), "id_03": (1.0,.89,.70),
            "id_04": (.92,.75,.57), "id_05": (.95,.90,.83), "id_06": (.99,.97,.83)},
 "pump":   {"id_00": (.84,.65,.58), "id_01": (.98,.90,.73), "id_02": (.45,.46,.52), "id_03": (.79,.81,.75),
            "id_04": (.99,.95,.93), "id_05": (.66,.66,.64), "id_06": (.94,.76,.61)},
 "slider": {"id_00": (.99,.99,.93), "id_01": (.94,.90,.83), "id_02": (.93,.79,.74), "id_03": (.99,.85,.71),
            "id_04": (.88,.78,.61), "id_05": (.84,.70,.60), "id_06": (.71,.56,.52)},
 "valve":  {"id_00": (.68,.55,.62), "id_01": (.77,.71,.61), "id_02": (.66,.59,.57), "id_03": (.70,.65,.44),
            "id_04": (.64,.65,.50), "id_05": (.52,.48,.44), "id_06": (.70,.66,.53)},
}
SNR_COLUMN = {6: 0, 0: 1, -6: 2}

## Step 12 - One full run

For one machine + one ID: get embeddings, split, score with every feature and both scorers.

In [ ]:
def run_one(machine, mid):
    files, emb = get_embeddings(machine, mid)
    index = {f: i for i, f in enumerate(files)}
    healthy, faulty = get_files(machine, mid)
    train_f, val_f, test_healthy_f = split_healthy(healthy)
    test_f = test_healthy_f + faulty
    y_test = np.array([0] * len(test_healthy_f) + [1] * len(faulty))      # 1 = faulty
    print(f"  clips: train {len(train_f)}, val {len(val_f)}, test {len(test_healthy_f)} healthy + {len(faulty)} faulty")

    tr = [index[f] for f in train_f]
    va = [index[f] for f in val_f]
    te = [index[f] for f in test_f]
    published = PUBLISHED[machine][mid][SNR_COLUMN[SNR]] if SNR in SNR_COLUMN else np.nan

    rows, clip_rows = [], []
    for feat in FEATURES:
        for scorer_name, scorer in SCORERS.items():
            train = emb[feat][tr]
            val_s  = scorer(train, emb[feat][va])
            test_s = scorer(train, emb[feat][te])
            m = get_metrics(val_s, test_s, y_test)
            rows.append({"machine": machine, "id": mid, "snr": SNR, "feature": feat, "scorer": scorer_name,
                         **m, "published_auc": published, "n_train": len(tr)})
            clip_rows.append(pd.DataFrame({
                "machine": machine, "id": mid, "snr": SNR, "feature": feat, "scorer": scorer_name,
                "file": test_f, "label": y_test, "score": test_s,
                "alarm": (test_s > m["threshold"]).astype(int)}))
    return rows, pd.concat(clip_rows), {"y": y_test, "emb": emb, "tr": tr, "va": va, "te": te}

## Step 13 - Run everything

Results are saved to `results_embed.csv` after **every** run.

In [ ]:
plan = []
for machine in MACHINES:
    for mid in IDS:
        healthy, faulty = get_files(machine, mid)
        if len(healthy) >= 30 and len(faulty) >= 5:
            plan.append((machine, mid))
        else:
            print("skip", machine, mid, "- not enough clips")
print(len(plan), "runs planned")

In [ ]:
all_rows, all_clips, KEEP = [], [], {}
start = time.time()

for number, (machine, mid) in enumerate(plan, 1):
    if (time.time() - start) / 3600 > MAX_HOURS:
        print("Time limit reached. Stopping.")
        break
    print(f"[{number}/{len(plan)}] {machine} {mid}  ({(time.time() - start) / 60:.1f} min so far)")
    rows, clips, keep = run_one(machine, mid)
    all_rows += rows
    all_clips.append(clips)
    KEEP[(machine, mid)] = keep
    pd.DataFrame(all_rows).to_csv(WORK_DIR + "/results_embed.csv", index=False)
    h = next(r for r in rows if (r["feature"], r["scorer"]) == HEADLINE)
    print(f"  headline AUC {h['auc']:.3f} | pAUC {h['pauc']:.3f} | recall {h['recall']:.3f} | false alarm {h['false_alarm']:.3f} | published AUC {h['published_auc']}")

res = pd.DataFrame(all_rows)
clip_table = pd.concat(all_clips, ignore_index=True)
clip_table.to_csv(WORK_DIR + "/clip_scores_embed.csv", index=False)
print(f"Done in {(time.time() - start) / 60:.1f} minutes. Runs:", len(plan))

## Results

### Main table (the choice made before running)

`HEADLINE` = feature `pool` + scorer `mahalanobis`. One row per run.

In [ ]:
head = res[(res.feature == HEADLINE[0]) & (res.scorer == HEADLINE[1])].copy()
cols = ["machine", "id", "auc", "pauc", "f1", "recall", "false_alarm", "recall_at_5pct_fa", "accuracy", "bal_acc", "published_auc"]
display(head[cols].round(3).reset_index(drop=True))
print("Mean over runs:")
display(head.groupby("machine")[["auc", "pauc", "recall", "false_alarm", "recall_at_5pct_fa", "accuracy", "bal_acc"]].mean().round(3))
print("Overall mean AUC:", round(head.auc.mean(), 3), "| published mean:", round(head.published_auc.mean(), 3))

### All variants

8 variants = 4 features x 2 scorers. **Careful:** picking the best row *after seeing the test result* is choosing on the test set. Use this table to understand, and report the main table above.

In [ ]:
variants = (res.groupby(["feature", "scorer"])[["auc", "pauc", "recall", "false_alarm", "recall_at_5pct_fa", "bal_acc"]]
              .mean().round(3))
display(variants)
by_machine = res.pivot_table(index="machine", columns=["feature", "scorer"], values="auc").round(3)
display(by_machine)

### Chart: mean AUC per machine, all variants

In [ ]:
ax = by_machine.T.plot(kind="bar", figsize=(11, 4), width=0.85)
ax.axhline(0.5, color="gray", ls="--", lw=1)
ax.set(ylabel="mean AUC", xlabel="feature, scorer", title="Mean AUC per machine", ylim=(0.3, 1.0))
ax.legend(title="machine", ncol=4)
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()

### Chart: AUC and recall at 5% false alarm (main variant)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for ax, col, title in zip(axs, ["auc", "recall_at_5pct_fa"], ["AUC", "Recall at 5% false alarm"]):
    t = head.pivot(index="machine", columns="id", values=col)
    im = ax.imshow(t.values, vmin=0, vmax=1, cmap="viridis")
    ax.set(xticks=range(t.shape[1]), xticklabels=t.columns, yticks=range(t.shape[0]), yticklabels=t.index, title=title)
    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            v = t.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="white" if v < 0.6 else "black")
plt.tight_layout(); plt.show()

### Chart: ours vs published AUC (main variant)

The published numbers are for the paper's own model. They are a reference, not a fair match.

In [ ]:
g = head.groupby("machine")[["auc", "published_auc"]].mean()
ax = g.plot(kind="bar", figsize=(7, 4), rot=0)
ax.axhline(0.5, color="gray", ls="--", lw=1)
ax.set(ylabel="mean AUC", ylim=(0.3, 1.0), title="Pretrained embeddings vs published"); plt.show()

### One run in detail

Change `KEY` to look at another run.

In [ ]:
KEY = ("valve", "id_00")          # <-- change this
if KEY not in KEEP:
    KEY = list(KEEP)[0]
    print("That run was not found. Using", KEY)

sub = clip_table[(clip_table.machine == KEY[0]) & (clip_table.id == KEY[1]) &
                 (clip_table.feature == HEADLINE[0]) & (clip_table.scorer == HEADLINE[1])]
m = head[(head.machine == KEY[0]) & (head.id == KEY[1])].iloc[0]
y, s = sub.label.values, sub.score.values

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
bins = np.linspace(s.min(), np.percentile(s, 99.5), 40)
axs[0].hist(s[y == 0], bins=bins, alpha=0.6, density=True, label="healthy (test)", color="tab:green")
axs[0].hist(s[y == 1], bins=bins, alpha=0.6, density=True, label="faulty (test)", color="tab:red")
axs[0].axvline(m.threshold, color="k", ls="--", label="threshold")
axs[0].set(title=f"Scores - {KEY}", xlabel="distance score"); axs[0].legend()
ConfusionMatrixDisplay(np.array([[m.tn, m.fp], [m.fn, m.tp]]), display_labels=["healthy", "faulty"]).plot(ax=axs[1], cmap="Blues", colorbar=False)
axs[1].set_title("Confusion matrix at the threshold")
plt.tight_layout(); plt.show()
print(f"false alarms {m.fp}/{m.fp + m.tn} | missed faults {m.fn}/{m.fn + m.tp} | recall {m.recall:.3f}")

## Save files

- `results_embed.csv` = one row per run and variant.
- `clip_scores_embed.csv` = one row per test clip and variant.
- `embeddings/` = saved embeddings (reuse them, no need to run AST again).
- `export_embed.zip` = the two CSV files, easy to download.

In [ ]:
os.makedirs(WORK_DIR + "/export_embed", exist_ok=True)
res.to_csv(WORK_DIR + "/export_embed/results_embed.csv", index=False)
clip_table.to_csv(WORK_DIR + "/export_embed/clip_scores_embed.csv", index=False)
shutil.make_archive(WORK_DIR + "/export_embed", "zip", WORK_DIR + "/export_embed")
print("Saved:", os.listdir(WORK_DIR + "/export_embed"))

## Summary to paste back

Run this cell and copy its output.

In [ ]:
print("=== MAIN VARIANT", HEADLINE, "===")
print(head[cols].round(3).to_csv(index=False))
print("=== MEAN PER MACHINE (main) ===")
print(head.groupby("machine")[["auc", "pauc", "recall", "false_alarm", "recall_at_5pct_fa", "accuracy", "bal_acc"]].mean().round(3).to_csv())
print("=== ALL VARIANTS, MEAN ===")
print(variants.to_csv())
print("=== AUC PER MACHINE AND VARIANT ===")
print(by_machine.to_csv())